In [49]:
import sys

from cv2.typing import map_int_and_double

sys.path.append('/home/simon/Work/morphometry')
import base64
import json
import numpy as np
import pyvista as pv
import nibabel as nib
from morphometry.image_io import Image
from matplotlib import pyplot as plt
from io import BytesIO
from tqdm import tqdm

In [2]:
patient = 'PA000464'

hip = Image('nibabel')
hip.read_image(f'/home/simon/Data/Augsburg_large/reader_study_augsburg/{patient}/hip.nii.gz')
knee = Image('nibabel')
knee.read_image(f'/home/simon/Data/Augsburg_large/reader_study_augsburg/{patient}/knee.nii.gz')
ankle = Image('nibabel')
ankle.read_image(f'/home/simon/Data/Augsburg_large/reader_study_augsburg/{patient}/ankle.nii.gz')

In [3]:
hip.transform_coordinate_system()
knee.transform_coordinate_system()
ankle.transform_coordinate_system()

In [36]:
concat = np.concatenate((hip.array, knee.array, ankle.array), axis=2)
layers = []
for i in tqdm(range(concat.shape[1])):
    fig =  plt.Figure()
    ax = fig.subplots()
    ax.imshow(concat[:, i, :].T, cmap='gray')
    ax.axis('off')
    ax.set_aspect(abs(hip.spacing[2]) / 2 * abs(hip.spacing[0]))

    buffer = BytesIO()
    fig.savefig(buffer, format='png')
    layers.append(base64.b64encode(buffer.getbuffer()).decode('ascii'))

100%|██████████| 324/324 [00:06<00:00, 48.47it/s]


In [37]:
d = {'layers': layers}
with open(f'/home/simon/Work/morphometry_ui/app/mad_ui_test.json', 'w') as f:
    json.dump(d, f)

In [4]:
hip_coords = np.argwhere(hip.array > 0)
knee_coords = np.argwhere(knee.array > 0)
ankle_coords = np.argwhere(ankle.array > 0)

In [5]:
def normalize_percentile(array, lower_percentile=1, upper_percentile=99):
    """Normalize based on percentiles to handle outliers"""
    non_zero_values = array[array > 0]

    p_low = np.percentile(non_zero_values, lower_percentile)
    p_high = np.percentile(non_zero_values, upper_percentile)

    normalized = np.clip((array - p_low) / (p_high - p_low), 0, 1)
    normalized[array == 0] = 0  # Keep background zeros

    return normalized

In [6]:
from skimage import exposure

def match_histograms(source, reference):
    """Match histogram of source to reference image"""
    non_zero_mask = source > 0
    ref_non_zero_mask = reference > 0

    matched = np.zeros_like(source, dtype=float)
    matched[non_zero_mask] = exposure.match_histograms(
        source[non_zero_mask],
        reference[ref_non_zero_mask]
    )

    return matched


In [7]:
def normalize_zscore(array):
    """Normalize using z-score (mean=0, std=1)"""
    non_zero_mask = array > 0
    mean_val = np.mean(array[non_zero_mask])
    std_val = np.std(array[non_zero_mask])

    normalized = np.zeros_like(array, dtype=float)
    normalized[non_zero_mask] = (array[non_zero_mask] - mean_val) / std_val

    return normalized

In [8]:
hip_normalised = normalize_zscore(hip.array)
knee_normalised = normalize_zscore(knee.array)
ankle_normalised = normalize_zscore(ankle.array)

In [9]:
array = np.zeros((hip.shape[0], hip.shape[1], hip.shape[2] + knee.shape[2] + ankle.shape[2]), dtype=np.uint8)
for coord in hip_coords:
    array[tuple(coord)] = hip_normalised[tuple(coord)]
for coord in knee_coords:
    array[tuple(coord[:2]) + (coord[2] + hip.shape[2],)] = knee_normalised[tuple(coord)]
for coord in ankle_coords:
    array[tuple(coord[:2]) + (coord[2] + hip.shape[2] + knee.shape[2],)] = ankle_normalised[tuple(coord)]

In [33]:
layers = []
for i in tqdm(range(array.shape[1])):
    # Get image dimensions
    img_slice = array[:, i, :].T
    height, width = img_slice.shape

    # Create figure with exact pixel dimensions (convert to inches for matplotlib)
    dpi = 100  # dots per inch
    fig = plt.Figure(figsize=(width/dpi, height/dpi), dpi=dpi)

    # Create axes that fill the entire figure
    ax = fig.add_axes([0, 0, 1, 1])  # [left, bottom, width, height] in figure coordinates
    ax.imshow(img_slice, cmap='gray')
    ax.axis('off')
    ax.set_aspect(abs(hip.spacing[2]) / 2 * abs(hip.spacing[0]))

    # Remove all padding and margins
    fig.subplots_adjust(left=0, bottom=0, right=1, top=1, wspace=0, hspace=0)

    buffer = BytesIO()
    fig.savefig(buffer, format='png', bbox_inches='tight', pad_inches=0, dpi=dpi)

    layers.append(base64.b64encode(buffer.getbuffer()).decode('ascii'))
    plt.close(fig)  # Free memory

d = {'layers': layers}
with open(f'/home/simon/Work/morphometry_ui/app/mad_ui_test.json', 'w') as f:
    json.dump(d, f)

100%|██████████| 324/324 [00:02<00:00, 114.64it/s]


In [10]:
hip_coords_world, knee_coords_world, ankle_coords_world = [], [], []
map_hip, map_knee, map_ankle = {}, {}, {}

for coord in hip_coords:
    hip_coords_world.append(hip.transform_index_to_physical_point(coord))
    map_hip[tuple(hip_coords_world[-1])] = tuple(coord)

for coord in knee_coords:
    knee_coords_world.append(knee.transform_index_to_physical_point(coord))
    map_knee[tuple(knee_coords_world[-1])] = tuple(coord)

for coord in ankle_coords:
    ankle_coords_world.append(ankle.transform_index_to_physical_point(coord))
    map_ankle[tuple(ankle_coords_world[-1])] = tuple(coord)

In [11]:
hip_coords_world = np.array(hip_coords_world)
knee_coords_world = np.array(knee_coords_world)
ankle_coords_world = np.array(ankle_coords_world)

In [12]:
min_x = min([min(hip_coords_world[:, 0]), min(knee_coords_world[:, 0]), min(ankle_coords_world[:, 0])])
max_x = max([max(hip_coords_world[:, 0]), max(knee_coords_world[:, 0]), max(ankle_coords_world[:, 0])])
min_y = min([min(hip_coords_world[:, 1]), min(knee_coords_world[:, 1]), min(ankle_coords_world[:, 1])])
max_y = max([max(hip_coords_world[:, 1]), max(knee_coords_world[:, 1]), max(ankle_coords_world[:, 1])])
min_z = min([min(hip_coords_world[:, 2]), min(knee_coords_world[:, 2]), min(ankle_coords_world[:, 2])])
max_z = max([max(hip_coords_world[:, 2]), max(knee_coords_world[:, 2]), max(ankle_coords_world[:, 2])])

In [13]:
min_x, max_x, min_y, max_y, min_z, max_z

(-208.216552734375,
 223.89173889160156,
 -197.32763671875,
 169.21646118164062,
 -110.00775146484375,
 803.0496215820312)

In [14]:
hip_array = np.zeros((int(max_x - min_x) + 1, hip.shape[1], hip.shape[2]), dtype=np.uint8)
knee_array = np.zeros((int(max_x - min_x) + 1, knee.shape[1], knee.shape[2]), dtype=np.uint8)
ankle_array = np.zeros((int(max_x - min_x) + 1, ankle.shape[1], ankle.shape[2]), dtype=np.uint8)

In [15]:
for coord in hip_coords:
    coord_world = hip.transform_index_to_physical_point(coord)
    hip_array[int(coord_world[0]) + int(abs(min_x)), coord[1], coord[2]] = hip_normalised[tuple(coord)]
for coord in knee_coords:
    coord_world = knee.transform_index_to_physical_point(coord)
    knee_array[int(coord_world[0]) + int(abs(min_x)), coord[1], coord[2]] = knee_normalised[tuple(coord)]
for coord in ankle_coords:
    coord_world = ankle.transform_index_to_physical_point(coord)
    ankle_array[int(coord_world[0]) + int(abs(min_x)), coord[1], coord[2]] = ankle_normalised[tuple(coord)]

In [42]:
layers = []
for i in tqdm(range(hip_array.shape[1])):
    fig = plt.Figure(figsize=(30, 10))
    ax = fig.subplots(nrows=3, ncols=1)
    ax[0].imshow(hip_array[:, i, :].T, cmap='gray')
    ax[1].imshow(knee_array[:, i, :].T, cmap='gray')
    ax[2].imshow(ankle_array[:, i, :].T, cmap='gray')
    ax[0].axis('off')
    ax[1].axis('off')
    ax[2].axis('off')
    ax[0].set_aspect(abs(hip.spacing[2]) / 2 * abs(hip.spacing[0]))
    ax[1].set_aspect(abs(knee.spacing[2]) / 2 * abs(knee.spacing[0]))
    ax[2].set_aspect(abs(ankle.spacing[2]) / 2 * abs(ankle.spacing[0]))

    buffer = BytesIO()
    fig.savefig(buffer, format='png')
    layers.append(base64.b64encode(buffer.getbuffer()).decode('ascii'))

100%|██████████| 324/324 [00:20<00:00, 15.64it/s]


In [43]:
d = {'layers': layers}
with open(f'/home/simon/Work/morphometry_ui/app/mad_ui_test.json', 'w') as f:
    json.dump(d, f)

In [53]:
array = np.zeros((int(max_x - min_x) + 1, int(max_y - min_y) + 1, int(max_z - min_z) + 1), dtype=np.uint8)
#

for coord in hip_coords_world:
    array[int(coord[0] + abs(min_x)), int(coord[1] + abs(min_y)), int(coord[2] + abs(min_z))] = hip_normalised[map_hip[tuple(coord)]]
for coord in knee_coords_world:
    array[int(coord[0] + abs(min_x)), int(coord[1] + abs(min_y)), int(coord[2] + abs(min_z))] = knee_normalised[map_knee[tuple(coord)]]
for coord in ankle_coords_world:
    array[int(coord[0] + abs(min_x)), int(coord[1] + abs(min_y)), int(coord[2] + abs(min_z))] = ankle_normalised[map_ankle[tuple(coord)]]

In [32]:
def remove_zero_slices(array):
    """Remove all-zero slices along each axis of a 3D array"""
    # Find non-zero slices along each axis
    non_zero_x = np.any(array, axis=(1, 2))  # Check along axis 0
    non_zero_y = np.any(array, axis=(0, 2))  # Check along axis 1
    non_zero_z = np.any(array, axis=(0, 1))  # Check along axis 2

    # Remove zero slices
    reduced_array = array[non_zero_x, :, :]
    reduced_array = reduced_array[:, non_zero_y, :]
    reduced_array = reduced_array[:, :, non_zero_z]

    return reduced_array

In [81]:
def remove_zero_slices_in_ranges(array):
    x = np.ones(array.shape[0])
    y = np.ones(array.shape[1])
    z = np.ones(array.shape[2])

    for i in range(array.shape[0]):
        if np.count_nonzero(array[i, :, :]) == 0:
            found_one, found_zero = False, False
            for j in range(10):
                if i + j < array.shape[0] and np.count_nonzero(array[i + j, :, :]) > 0:
                    found_one = True
                if i - j >= 0 and np.count_nonzero(array[i - j, :, :]) > 0:
                    found_zero = True

            if found_one and found_zero:
                x[i] = 0

    for i in range(array.shape[1]):
        if np.count_nonzero(array[:, i, :]) == 0:
            found_one, found_zero = False, False
            for j in range(10):
                if i + j < array.shape[1] and np.count_nonzero(array[:, i + j, :]) > 0:
                    found_one = True
                if i - j >= 0 and np.count_nonzero(array[:, i - j, :]) > 0:
                    found_zero = True
            if found_one and found_zero:
                y[i] = 0

    for i in range(array.shape[2]):
        if np.count_nonzero(array[:, :, i]) == 0:
            found_one, found_zero = False, False
            for j in range(10):
                if i + j < array.shape[2] and np.count_nonzero(array[:, :, i + j]) > 0:
                    found_one = True
                if i - j >= 0 and np.count_nonzero(array[:, :, i - j]) > 0:
                    found_zero = True
            if found_one and found_zero:
                z[i] = 0

    x = x.astype(bool)
    y = y.astype(bool)
    z = z.astype(bool)

    return array[x][:, y][:, :, z]

In [82]:
array_reduced = remove_zero_slices_in_ranges(array)

In [83]:
array.shape, array_reduced.shape

((433, 367, 914), (422, 357, 467))

In [44]:
array = normalize_zscore(array)

In [84]:
layers = []
for i in tqdm(range(array.shape[1])):
    fig = plt.Figure()
    ax = fig.subplots()
    ax.imshow(array[:, i, :].T, cmap='gray')
    # ax.invert_yaxis()
    ax.axis('off')
    # ax.set_aspect(abs(hip.spacing[2]) / 2 * abs(hip.spacing[0]))

    buffer = BytesIO()
    fig.savefig(buffer, format='png')
    layers.append(base64.b64encode(buffer.getbuffer()).decode('ascii'))

100%|██████████| 367/367 [00:05<00:00, 71.29it/s]


In [85]:
layers = []
for i in tqdm(range(array.shape[2])):
    fig = plt.Figure()
    ax = fig.subplots()
    ax.imshow(array[:, :, 2].T, cmap='gray')
    # ax.invert_yaxis()
    ax.axis('off')
    # ax.set_aspect(abs(hip.spacing[2]) / 2 * abs(hip.spacing[0]))

    buffer = BytesIO()
    fig.savefig(buffer, format='png')
    layers.append(base64.b64encode(buffer.getbuffer()).decode('ascii'))

100%|██████████| 914/914 [00:11<00:00, 76.31it/s]


In [79]:
d = {'layers': layers}
with open(f'/home/simon/Work/morphometry_ui/app/mad_ui_test.json', 'w') as f:
    json.dump(d, f)

In [ ]:
p = pv.Plotter()
p.add_volume(array, cmap='gray')
p.show()

In [86]:
nib_image = nib.Nifti1Image(array, hip.affine, hip.header)
image = Image.from_nibabel(nib_image)
image.save_image('/home/simon/Downloads/test.nii.gz')